# CNN + Uncertainty Quantification (MC-Dropout) — CYGNO

Self-contained notebook for **Google Colab**. Trains the CNN with **MC-Dropout**
(epistemic uncertainty) in **three configurations** and displays **all the plots and
comparisons inline**. Nothing is written to disk — every result is shown in the notebook.

## The three experiments

| # | Experiment | Bags | Labels | Theoretical ceiling |
|---|------------|------|--------|---------------------|
| 1 | **Supervised MC** | Pure ER vs pure NR | True (per event) | ~0.89 (physical separability) |
| 2 | **CWoLa MC** | STD-like (100% ER) vs AmBe-like (64% ER + 36% NR) | Per bag only | 0.68 (bag-vs-bag) |
| 3 | **CWoLa real** | Real STD vs real AmBe | Per bag only | 0.68 (bag-vs-bag) |

The **CWoLa ceiling** comes from `AUC_max = 0.5 + 0.5*alpha` with `alpha = 0.36` (NR fraction
in AmBe) -> `0.68`. See `Dataset_Preparation/Fraction_Estimate.ipynb`.

The CNN architecture is **identical** to `build_cwola_classifier` from
[AML-Project](https://github.com/diegomonino/AML-Project) (`AML PROJECT.ipynb`). The only
change is that the two `Dropout` layers are replaced by `MCDropout` (dropout kept active at
inference time) so we can estimate epistemic uncertainty — architecturally identical.

## Before running on Colab
1. `Runtime > Change runtime type > GPU` (T4 is enough).
2. `Runtime > Run all`. The notebook clones the data repo automatically.

## 1. Setup: clone the repo and unzip the Monte Carlo dataset

In [1]:
# Cross-platform setup (pure Python: works on Colab / Kaggle / local).
import os
import subprocess
import zipfile

# Clone the repo that ships the images (idempotent).
if not os.path.exists("AML-Project"):
    print("Cloning repository...")
    subprocess.run(["git", "clone", "https://github.com/diegomonino/AML-Project.git"], check=True)
else:
    print("Repo already cloned.")

# Unzip the Monte Carlo dataset (pure ER / NR).
if not os.path.exists("ER_label_0"):
    zip_path = os.path.join("AML-Project", "Monte Carlo PURE", "dataset_train_bilanciato.zip")
    print(f"Unzipping {zip_path}...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(".")
    print("Done.")
else:
    print("MC dataset already unzipped.")

Repo already cloned.
MC dataset already unzipped.


## 2. Imports and global configuration

In [3]:
import glob
import gc
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report

# --- Global hyperparameters (matched to AML-Project / build_cwola_classifier) ---
IMG_SIZE = (128, 128)
VALIDATION_SPLIT = 0.2
SEED = 42
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 100
PATIENCE = 10
N_MC_PASSES = 50          # stochastic MC-Dropout passes at inference

# --- CWoLa physics ---
ALPHA = 0.36              # NR fraction in AmBe (from Fraction_Estimate.ipynb)
CWOLA_CEILING = 0.5 + 0.5 * ALPHA   # = 0.68 -> bag-vs-bag AUC ceiling

# Reproducibility
tf.random.set_seed(SEED)
np.random.seed(SEED)

# GPU
gpus = tf.config.list_physical_devices("GPU")
print("GPU available:", gpus if gpus else "NONE  (enable GPU in Runtime > Change runtime type)")
print(f"CWoLa theoretical ceiling (alpha={ALPHA}): AUC_max = {CWOLA_CEILING:.3f}")

GPU available: NONE  (enable GPU in Runtime > Change runtime type)
CWoLa theoretical ceiling (alpha=0.36): AUC_max = 0.680


## 3. Data utilities

In [ ]:
def load_all_images(folder, img_size=(128, 128)):
    """Load grayscale images from a folder, normalize to [0,1], skip corrupted ones."""
    exts = ["*.png", "*.PNG", "*.jpg", "*.jpeg", "*.bmp", "*.gif"]
    paths = []
    for e in exts:
        paths.extend(glob.glob(os.path.join(folder, e)))
    paths = sorted(set(paths))  # set() removes duplicates from overlapping extensions
    imgs = []
    for p in paths:
        try:
            im = Image.open(p).convert("L").resize(img_size, Image.BILINEAR)
            imgs.append(np.array(im, dtype=np.float32)[..., None] / 255.0)
        except Exception as ex:
            print(f"[load_all_images] skipping {p}: {ex}")
    if not imgs:
        raise RuntimeError(f"No images loaded from {folder}")
    return np.stack(imgs, axis=0)  # (N, H, W, 1)


def augment_tf(img):
    """On-the-fly augmentation: random flips + 90-degree rotations (same as reference)."""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    return tf.image.rot90(img, k)


def image_energy(images):
    """Energy proxy: per-image intensity sum (total deposited light)."""
    return images.sum(axis=(1, 2, 3))


def visualize_grid(images, titles=None, ncol=5, figsize=(12, 3)):
    n = len(images)
    nrow = int(np.ceil(n / ncol))
    plt.figure(figsize=figsize)
    for i in range(n):
        plt.subplot(nrow, ncol, i + 1)
        plt.imshow(images[i].squeeze(), cmap="gray", vmin=0, vmax=1)
        plt.axis("off")
        if titles:
            plt.title(titles[i], fontsize=8)
    plt.tight_layout()
    plt.show()

## 4. CNN model with `MCDropout`

`MCDropout` is a `Dropout` layer that stays **active at inference time too**. Each `predict`
is a stochastic sample -> the variance across many passes is the epistemic uncertainty.

`BatchNormalization` is NOT forced into training mode: calling `model(x, training=False)`
makes BN use its moving averages (correct) while dropout remains stochastic.

The convolutional stack, channel widths, strides, global pooling and dense heads are
**identical** to `build_cwola_classifier` in the reference notebook.

In [ ]:
class MCDropout(layers.Dropout):
    """Dropout that stays active at inference (Monte Carlo Dropout)."""
    def call(self, inputs):
        return super().call(inputs, training=True)


def build_classifier(input_shape=(128, 128, 1), lr=LEARNING_RATE):
    """CNN identical to AML-Project's build_cwola_classifier, with MC-Dropout heads."""
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, strides=1, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, strides=1, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, 3, strides=1, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = MCDropout(0.4)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = MCDropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, out, name="cnn_mcdropout")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[keras.metrics.AUC(name="roc_auc")],
    )
    return model


def make_dataset(X, y, batch_size, augment=False, shuffle_data=True):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle_data:
        ds = ds.shuffle(buffer_size=min(len(X), 8192), seed=SEED)
    if augment:
        ds = ds.map(lambda im, lb: (augment_tf(im), lb), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def train_model(X_train, y_train, X_val, y_val, name):
    """Train the CNN and return (model, history)."""
    print(f"\n=== Training: {name} ===")
    model = build_classifier(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 1), lr=LEARNING_RATE)
    train_ds = make_dataset(X_train, y_train, BATCH_SIZE, augment=True)
    val_ds = make_dataset(X_val, y_val, BATCH_SIZE, augment=False, shuffle_data=False)
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_roc_auc", patience=PATIENCE,
                                      mode="max", restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                          patience=4, verbose=1),
    ]
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                        callbacks=callbacks, verbose=1)
    return model, history


def mc_predict(model, X, n_passes=N_MC_PASSES, batch_size=256):
    """Return (mu, sigma) per image via N stochastic passes with dropout active."""
    n = len(X)
    preds = np.zeros((n_passes, n), dtype=np.float32)
    for t in range(n_passes):
        out = []
        for i in range(0, n, batch_size):
            xb = X[i:i + batch_size]
            out.append(model(xb, training=False).numpy().ravel())  # MCDropout stays active
        preds[t] = np.concatenate(out)
    return preds.mean(axis=0), preds.std(axis=0)

## 5. Evaluation function (everything shown inline, nothing saved)

In [ ]:
def save_training_curves(history):
    """Plot loss and AUC curves (train vs val)."""
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(history.history["loss"], label="train")
    ax[0].plot(history.history["val_loss"], label="val")
    ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=0.3)
    ax[1].plot(history.history["roc_auc"], label="train")
    ax[1].plot(history.history["val_roc_auc"], label="val")
    ax[1].set_title("AUC"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def evaluate_experiment(name, model, X_test, y_test, energy_test,
                        ceiling=None, class_names=("class 0", "class 1")):
    """MC-Dropout inference + all metrics and plots, displayed inline."""
    print(f"########## EVALUATION: {name} ##########")

    # --- MC-Dropout inference ---
    mu, sigma = mc_predict(model, X_test)
    print(f"mean score={mu.mean():.4f} | mean uncertainty={sigma.mean():.4f}")

    # --- Global AUC ---
    fpr, tpr, _ = roc_curve(y_test, mu)
    test_auc = auc(fpr, tpr)
    print(f"Test AUC (MC mean): {test_auc:.4f}")
    if ceiling is not None:
        print(f"CWoLa theoretical ceiling: {ceiling:.4f}  ->  {100*test_auc/ceiling:.1f}% of ceiling")

    # --- Confusion matrix + report ---
    y_pred = (mu > 0.5).astype(int)
    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nReport:")
    print(classification_report(y_test, y_pred, target_names=list(class_names), zero_division=0))

    # --- ROC ---
    plt.figure(figsize=(5, 5))
    plt.plot(fpr, tpr, label=f"AUC={test_auc:.3f}")
    plt.plot([0, 1], [0, 1], "k--", alpha=0.3, label="chance (0.5)")
    ttl = f"ROC - {name}"
    if ceiling is not None:
        ttl += f"\n(CWoLa ceiling AUC={ceiling:.2f})"
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(ttl); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    # --- Score distribution ---
    plt.figure(figsize=(8, 4))
    plt.hist(mu[y_test == 0], bins=40, alpha=0.6, label=class_names[0])
    plt.hist(mu[y_test == 1], bins=40, alpha=0.6, label=class_names[1])
    plt.xlabel("score (MC mean)"); plt.ylabel("frequency")
    plt.title(f"Score distribution - {name}"); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    # --- Uncertainty vs energy ---
    plt.figure(figsize=(8, 5))
    plt.scatter(energy_test[y_test == 0], sigma[y_test == 0], s=5, alpha=0.3, label=class_names[0])
    plt.scatter(energy_test[y_test == 1], sigma[y_test == 1], s=5, alpha=0.3, label=class_names[1])
    plt.xlabel("Energy (pixel sum)"); plt.ylabel("Uncertainty sigma (MC-Dropout)")
    plt.title(f"Uncertainty vs Energy - {name}"); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    # mean sigma per energy bin
    ebins = np.percentile(energy_test, np.linspace(0, 100, 11))
    centers, sig_means = [], []
    for lo, hi in zip(ebins[:-1], ebins[1:]):
        m = (energy_test >= lo) & (energy_test < hi)
        if m.sum():
            centers.append((lo + hi) / 2); sig_means.append(sigma[m].mean())
    plt.figure(figsize=(8, 4))
    plt.plot(centers, sig_means, "o-")
    plt.xlabel("Energy (bin center)"); plt.ylabel("mean sigma")
    plt.title(f"Mean uncertainty vs energy - {name}"); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    # --- Confidence cut ---
    print("\nConfidence cut (keep the most certain X%):")
    print(f"{'% most certain':>14} {'N used':>10} {'AUC':>8}")
    for frac in [1.0, 0.9, 0.7, 0.5, 0.3]:
        thr = np.quantile(sigma, frac)
        m = sigma <= thr
        if len(np.unique(y_test[m])) > 1:
            f, t, _ = roc_curve(y_test[m], mu[m])
            print(f"{int(frac*100):>13}% {int(m.sum()):>10} {auc(f, t):>8.4f}")

    # --- AUC per energy bin ---
    print("\nAUC per energy bin:")
    print(f"{'energy bin':>26} {'N':>6} {'AUC':>8} {'sigma':>9}")
    ebins5 = np.percentile(energy_test, np.linspace(0, 100, 6))
    bin_centers, bin_aucs = [], []
    for lo, hi in zip(ebins5[:-1], ebins5[1:]):
        m = (energy_test >= lo) & (energy_test <= hi)
        if m.sum() and len(np.unique(y_test[m])) > 1:
            f, t, _ = roc_curve(y_test[m], mu[m])
            a = auc(f, t)
            bin_centers.append((lo + hi) / 2); bin_aucs.append(a)
            print(f"[{lo:8.1f}, {hi:8.1f}] {int(m.sum()):>6} {a:>8.4f} {sigma[m].mean():>9.4f}")

    # AUC vs energy plot
    if bin_aucs:
        plt.figure(figsize=(8, 4))
        plt.plot(bin_centers, bin_aucs, "o-", label="AUC per bin")
        plt.axhline(0.5, color="k", ls="--", alpha=0.3, label="chance")
        if ceiling is not None:
            plt.axhline(ceiling, color="r", ls="--", alpha=0.5, label=f"CWoLa ceiling {ceiling:.2f}")
        plt.xlabel("Energy (bin center)"); plt.ylabel("AUC")
        plt.title(f"AUC vs Energy - {name}"); plt.legend(); plt.grid(alpha=0.3)
        plt.ylim(0.4, 1.0); plt.tight_layout(); plt.show()

    return {"name": name, "test_auc": float(test_auc),
            "ceiling": (float(ceiling) if ceiling is not None else None),
            "mu_mean": float(mu.mean()), "sigma_mean": float(sigma.mean())}

## 6. Load the image pools (MC and real)

In [ ]:
print("Loading MC pools (pure ER / NR)...")
er_pool = load_all_images("ER_label_0", img_size=IMG_SIZE)   # simulated ER
nr_pool = load_all_images("NR_label_1", img_size=IMG_SIZE)   # simulated NR
print(f"  MC ER: {len(er_pool)} | MC NR: {len(nr_pool)}")

print("Loading real data (STD / AmBe, train and test)...")
RBASE = "AML-Project/Dataset_Preparation"
std_train  = load_all_images(f"{RBASE}/STD_images_train",  img_size=IMG_SIZE)
ambe_train = load_all_images(f"{RBASE}/AmBe_images_train", img_size=IMG_SIZE)
std_test   = load_all_images(f"{RBASE}/STD_images_test",   img_size=IMG_SIZE)
ambe_test  = load_all_images(f"{RBASE}/AmBe_images_test",  img_size=IMG_SIZE)
print(f"  STD train: {len(std_train)} | AmBe train: {len(ambe_train)}")
print(f"  STD test:  {len(std_test)} | AmBe test:  {len(ambe_test)}")

# Sample preview
print("Sample MC ER:");  visualize_grid(er_pool[:5],   titles=[f"ER-{i}" for i in range(5)])
print("Sample MC NR:");  visualize_grid(nr_pool[:5],   titles=[f"NR-{i}" for i in range(5)])
print("Sample real STD:");  visualize_grid(std_train[:5],  titles=[f"STD-{i}" for i in range(5)])
print("Sample real AmBe:"); visualize_grid(ambe_train[:5], titles=[f"AmBe-{i}" for i in range(5)])

results = {}  # accumulates each experiment's dict

## 7. Experiment 1 - Supervised CNN on MC (ER vs NR, with labels)

True per-event labels. This is the **physical ceiling** reference: how well the CNN separates
ER/NR when the classes are clean. Expected AUC ~0.85-0.90.

In [ ]:
# Split train/val/test over pure ER/NR with true labels.
X1 = np.concatenate([er_pool, nr_pool], axis=0)
y1 = np.concatenate([np.zeros(len(er_pool), np.float32),
                     np.ones(len(nr_pool), np.float32)], axis=0)
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(X1))
X1, y1 = X1[perm], y1[perm]

n_test = int(len(X1) * 0.15)
n_val  = int(len(X1) * 0.15)
X1_test,  y1_test  = X1[:n_test], y1[:n_test]
X1_val,   y1_val   = X1[n_test:n_test + n_val], y1[n_test:n_test + n_val]
X1_train, y1_train = X1[n_test + n_val:], y1[n_test + n_val:]
print(f"Train: {len(X1_train)} | Val: {len(X1_val)} | Test: {len(X1_test)}")

model1, hist1 = train_model(X1_train, y1_train, X1_val, y1_val, "exp1_MC_supervised")
save_training_curves(hist1)

energy1_test = image_energy(X1_test)
results["exp1"] = evaluate_experiment(
    "exp1_MC_supervised", model1, X1_test, y1_test, energy1_test,
    ceiling=None, class_names=("ER", "NR"))

del X1, y1, X1_train, X1_val, X1_test, model1; gc.collect()

## 8. Experiment 2 - CWoLa on MC (STD-like 100% ER vs AmBe-like 36% NR)

Bags with realistic proportions, labeled **per bag only** (not per event). Evaluated two ways:
- **Bag-vs-bag** (held-out MC mixtures): theoretical ceiling **0.68**.
- **Physics** (pure ER vs NR, true labels): measures whether CWoLa recovered the real separation.

The train, test-bag and test-physics images are **disjoint** (no data leakage).

In [ ]:
# Shuffle pools and use cursors to guarantee disjoint sets.
rng = np.random.default_rng(SEED)
er = er_pool[rng.permutation(len(er_pool))]
nr = nr_pool[rng.permutation(len(nr_pool))]
ec, nc = 0, 0

# (a) Physics test: pure ER/NR with true labels.
N_PHYS = 800
er_phys = er[ec:ec + N_PHYS]; ec += N_PHYS
nr_phys = nr[nc:nc + N_PHYS]; nc += N_PHYS

TRAIN_BAG = 2000
TEST_BAG  = 700

# (b) Training bags.
tr_bag0 = er[ec:ec + TRAIN_BAG]; ec += TRAIN_BAG                  # STD-like: 100% ER
n_er_b1 = int(round(TRAIN_BAG * (1 - ALPHA)))
n_nr_b1 = TRAIN_BAG - n_er_b1
tr_bag1 = np.concatenate([er[ec:ec + n_er_b1], nr[nc:nc + n_nr_b1]], axis=0)  # AmBe-like
ec += n_er_b1; nc += n_nr_b1

# (c) Test bags (held-out, same proportions).
te_bag0 = er[ec:ec + TEST_BAG]; ec += TEST_BAG
n_er_tb1 = int(round(TEST_BAG * (1 - ALPHA)))
n_nr_tb1 = TEST_BAG - n_er_tb1
te_bag1 = np.concatenate([er[ec:ec + n_er_tb1], nr[nc:nc + n_nr_tb1]], axis=0)
ec += n_er_tb1; nc += n_nr_tb1

print(f"ER used: {ec}/{len(er)} | NR used: {nc}/{len(nr)}")
print(f"Train bag0 (STD-like): {len(tr_bag0)} (100% ER)")
print(f"Train bag1 (AmBe-like): {len(tr_bag1)} ({n_er_b1} ER + {n_nr_b1} NR = {100*n_nr_b1/TRAIN_BAG:.0f}% NR)")

# Label = bag (0 = STD-like, 1 = AmBe-like).
Xb = np.concatenate([tr_bag0, tr_bag1], axis=0)
yb = np.concatenate([np.zeros(len(tr_bag0), np.float32),
                     np.ones(len(tr_bag1), np.float32)], axis=0)
perm = rng.permutation(len(Xb))
Xb, yb = Xb[perm], yb[perm]
n_val = int(len(Xb) * VALIDATION_SPLIT)
X2_val,   y2_val   = Xb[:n_val], yb[:n_val]
X2_train, y2_train = Xb[n_val:], yb[n_val:]
print(f"Train: {len(X2_train)} | Val: {len(X2_val)}")

model2, hist2 = train_model(X2_train, y2_train, X2_val, y2_val, "exp2_CWoLa_MC")
save_training_curves(hist2)

In [ ]:
# Evaluation (a): bag-vs-bag (ceiling 0.68).
X2_test_bag = np.concatenate([te_bag0, te_bag1], axis=0)
y2_test_bag = np.concatenate([np.zeros(len(te_bag0), np.float32),
                              np.ones(len(te_bag1), np.float32)], axis=0)
energy2_bag = image_energy(X2_test_bag)
results["exp2_bag"] = evaluate_experiment(
    "exp2_CWoLa_MC_bag", model2, X2_test_bag, y2_test_bag, energy2_bag,
    ceiling=CWOLA_CEILING, class_names=("STD-like", "AmBe-like"))

# Evaluation (b): physics ER vs NR (true labels).
X2_test_phys = np.concatenate([er_phys, nr_phys], axis=0)
y2_test_phys = np.concatenate([np.zeros(len(er_phys), np.float32),
                               np.ones(len(nr_phys), np.float32)], axis=0)
energy2_phys = image_energy(X2_test_phys)
results["exp2_phys"] = evaluate_experiment(
    "exp2_CWoLa_MC_physics", model2, X2_test_phys, y2_test_phys, energy2_phys,
    ceiling=None, class_names=("ER", "NR"))

del Xb, X2_train, X2_val, model2; gc.collect()

## 9. Experiment 3 - CWoLa on real data (STD vs AmBe)

The real physics experiment. No per-event labels: we only know which run each image came from.
Theoretical ceiling **0.68**. Expected AUC ~0.60 (lower because of real noise + weak signal).

In [ ]:
# Real bags, label = run (STD=0, AmBe=1).
X3 = np.concatenate([std_train, ambe_train], axis=0)
y3 = np.concatenate([np.zeros(len(std_train), np.float32),
                     np.ones(len(ambe_train), np.float32)], axis=0)
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(X3))
X3, y3 = X3[perm], y3[perm]
n_val = int(len(X3) * VALIDATION_SPLIT)
X3_val,   y3_val   = X3[:n_val], y3[:n_val]
X3_train, y3_train = X3[n_val:], y3[n_val:]
print(f"Train: {len(X3_train)} | Val: {len(X3_val)}")

model3, hist3 = train_model(X3_train, y3_train, X3_val, y3_val, "exp3_CWoLa_real")
save_training_curves(hist3)

# Test = held-out real STD/AmBe.
X3_test = np.concatenate([std_test, ambe_test], axis=0)
y3_test = np.concatenate([np.zeros(len(std_test), np.float32),
                          np.ones(len(ambe_test), np.float32)], axis=0)
energy3_test = image_energy(X3_test)
results["exp3"] = evaluate_experiment(
    "exp3_CWoLa_real", model3, X3_test, y3_test, energy3_test,
    ceiling=CWOLA_CEILING, class_names=("STD", "AmBe"))

del X3, y3, X3_train, X3_val, X3_test, model3; gc.collect()

## 10. Final comparison

In [ ]:
# Summary table.
rows = [
    ("Exp 1 - MC supervised (ER vs NR)",     results["exp1"]["test_auc"],      None),
    ("Exp 2 - CWoLa MC bag-vs-bag",          results["exp2_bag"]["test_auc"],  CWOLA_CEILING),
    ("Exp 2 - CWoLa MC physics (ER vs NR)",  results["exp2_phys"]["test_auc"], None),
    ("Exp 3 - CWoLa real (STD vs AmBe)",     results["exp3"]["test_auc"],      CWOLA_CEILING),
]

print("=" * 64)
print("COMPARATIVE SUMMARY")
print("=" * 64)
print(f"{'Experiment':<40} {'AUC':>7} {'Ceiling':>8}")
print("-" * 64)
for label, a, c in rows:
    cs = f"{c:.3f}" if c is not None else "  -  "
    print(f"{label:<40} {a:>7.4f} {cs:>8}")
print("=" * 64)

# Bar chart.
labels = [r[0].replace(" - ", "\n") for r in rows]
aucs = [r[1] for r in rows]
plt.figure(figsize=(10, 5))
plt.bar(range(len(aucs)), aucs, color=["#2ca02c", "#1f77b4", "#17becf", "#ff7f0e"])
plt.axhline(0.5, color="k", ls="--", alpha=0.4, label="chance (0.5)")
plt.axhline(CWOLA_CEILING, color="r", ls="--", alpha=0.6, label=f"CWoLa ceiling ({CWOLA_CEILING:.2f})")
for i, a in enumerate(aucs):
    plt.text(i, a + 0.01, f"{a:.3f}", ha="center", fontsize=10)
plt.xticks(range(len(aucs)), labels, fontsize=8)
plt.ylabel("Test AUC"); plt.ylim(0.4, 1.0)
plt.title("Comparison of the three trainings")
plt.legend(); plt.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

## 11. Reading the results

- **Exp 1 (supervised, ~0.89)**: physical ceiling. The CNN separates ER/NR when labels are clean.
- **Exp 2 bag-vs-bag (~0.67)**: honest CWoLa on MC. Should stay **below 0.68** (the theoretical
  ceiling with alpha=0.36). If it goes above, the proportions or the split have a leak.
- **Exp 2 physics**: measures how much of the real ER/NR separation CWoLa recovered while training
  on bags only. The closer to the 0.89 supervised number, the better it learned the physics.
- **Exp 3 (real, ~0.60)**: the real experiment. Below the 0.68 ceiling because of detector noise
  and weak signal. The way to push it up is to **clean the data** (zero-suppression + GNN), not
  more epochs or a different architecture.

Everything is displayed inline; nothing is written to disk in this version.